In [2]:
import requests
from bs4 import BeautifulSoup
from datetime import datetime, timedelta
import csv
import time

today = datetime.today().date()
base_url = "https://wuzzuf.net/search/jobs"

In [3]:
page_num = 0
jobs = []
while True:
    url = f"{base_url}?a=spbg&q=data%20engineer&start={page_num}"
    page = requests.get(url, headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"})
    soup = BeautifulSoup(page.content, "html.parser")
    
    results = soup.find_all(class_="css-ghe2tq e1v1l3u10")
    
    # Stop if no results found on this page
    if not results:
        break
    
    for job in results:
        try:
            title = job.find(class_="css-193uk2c").text.strip()
            companyName = job.find(class_="css-ipsyv7").text.strip("-")
            location = job.find(class_="css-16x61xq").text.strip()
            jobType = job.find(class_="css-uc9rga eoyjyou0").text.strip()
            experienceLevel = job.find(class_="css-1rhj4yg").find("a", class_="css-o171kl").text.strip()
            postingDate = job.find(class_="css-eg55jf") or job.find(class_="css-1jldrig")
            postingDate = postingDate.text.strip()

            #Date Convertion
            if "days ago" in postingDate:
                siteDays = int(postingDate.split()[0])
                actualDate = today - timedelta(days=siteDays)
            else:
                actualDate = today

            jobs.append({
                "title": title,
                "companyName": companyName,
                "location": location,
                "jobType": jobType,
                "experience": experienceLevel,
                "Date": actualDate
            })
        except AttributeError:
            # Skip jobs with missing fields
            continue

    print(f"Page {page_num + 1} scraped — {len(results)} jobs found")
    page_num += 1
    time.sleep(1)  #Avoids getting blocked by the site for running too many requests

with open("Raw.csv", "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=["title", "companyName", "location", "jobType", "experience", "Date"])
    writer.writeheader()
    writer.writerows(jobs)

print(f"Done! Total jobs saved: {len(jobs)}")

Page 1 scraped — 15 jobs found
Page 2 scraped — 15 jobs found
Page 3 scraped — 15 jobs found
Page 4 scraped — 15 jobs found
Page 5 scraped — 15 jobs found
Page 6 scraped — 15 jobs found
Page 7 scraped — 15 jobs found
Page 8 scraped — 15 jobs found
Page 9 scraped — 15 jobs found
Page 10 scraped — 15 jobs found
Page 11 scraped — 15 jobs found
Page 12 scraped — 15 jobs found
Page 13 scraped — 15 jobs found
Page 14 scraped — 15 jobs found
Page 15 scraped — 15 jobs found
Page 16 scraped — 15 jobs found
Page 17 scraped — 15 jobs found
Page 18 scraped — 15 jobs found
Page 19 scraped — 15 jobs found
Page 20 scraped — 15 jobs found
Page 21 scraped — 15 jobs found
Page 22 scraped — 15 jobs found
Page 23 scraped — 15 jobs found
Page 24 scraped — 15 jobs found
Page 25 scraped — 15 jobs found
Page 26 scraped — 15 jobs found
Page 27 scraped — 15 jobs found
Page 28 scraped — 15 jobs found
Page 29 scraped — 15 jobs found
Page 30 scraped — 10 jobs found
Done! Total jobs saved: 445


In [5]:
###cleaning
import pandas as pd

df=pd.read_csv('Raw.csv')

df["Date"]=pd.to_datetime(df['Date'])

print(df.duplicated().sum())

df=df.fillna("Unkown")

print(df.info())

df.to_csv("cleaned.csv", index=False, encoding="utf-8-sig")    


0
<class 'pandas.DataFrame'>
RangeIndex: 445 entries, 0 to 444
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   title        445 non-null    str           
 1   companyName  445 non-null    str           
 2   location     445 non-null    str           
 3   jobType      445 non-null    str           
 4   experience   445 non-null    str           
 5   Date         445 non-null    datetime64[us]
dtypes: datetime64[us](1), str(5)
memory usage: 21.0 KB
None
